## Configuration and Imports

In [ ]:
import json
import re
import os

# --- CONFIGURATION ---
INPUT_FILE = "../data/raw/metamath_base_no_instruct_eval.json"
OUTPUT_FILE = "../data/results/metamath_base_no_instruct_eval_test.json"
MAX_ID = -1  # Set to -1 to process all records


## Extract EDA Logic

In [ ]:
def clean_math_string(s: str) -> str:
    """Normalizes math outputs: safely unwraps \boxed{}, strips $, cleans commas, parses sentences, and truncates loops."""
    if not isinstance(s, str):
        return str(s)

    s = s.strip()

    # 1. Truncate generation collapse loops (newlines, asterisks)
    s = s.split("\n")[0]
    s = s.split("*")[0]

    # 2. Remove non-ASCII spam
    s = re.sub(r"[^\x00-\x7F]+", "", s)

    # Remove wrapping #### or spaces
    s = re.sub(r"^#+", "", s)
    s = re.sub(r"#+$", "", s)
    s = s.strip()

    # Safely extract from \boxed{...} handling nested brackets
    if "\\boxed{" in s:
        start = s.find("\\boxed{") + 7
        depth = 1
        for i in range(start, len(s)):
            if s[i] == "{":
                depth += 1
            elif s[i] == "}":
                depth -= 1
            if depth == 0:
                s = s[start:i]
                break

    s = s.strip("$ \n\t.")

    # Remove conversational prefixes
    s = re.sub(r"^(The )?(final )?answer is\s*:?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^Therefore,?\s*(the )?.*?\s+is\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^So,?\s*(the )?.*?\s+is\s*", "", s, flags=re.IGNORECASE)

    s = s.strip("$ \n\t.")

    # Sentence parsing fallback
    if len(s) > 10 and not any(
        op in s for op in ["+", "-", "*", "/", "\\", "^", "=", "x", "y", "z"]
    ):
        nums = re.findall(r"-?[\d,]+(?:\.\d+)?", s)
        if nums:
            s = nums[-1]

    # Remove commas
    if re.match(r"^-?[\d,]+(?:\.\d+)?$", s):
        s = s.replace(",", "")

    return s.strip()


def extract_model_answer(text: str) -> str:
    ans = "Extraction Failed"

    if "####" in text:
        parts = text.split("####")
        ans = parts[1] if len(parts) > 1 and parts[1].strip() else parts[-1]
    elif "The final answer is:" in text:
        ans = text.split("The final answer is:")[-1]
    elif "The answer is:" in text:
        ans = text.split("The answer is:")[-1]
    else:
        numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", text)
        if numbers:
            ans = numbers[-1]

    if ans == "Extraction Failed":
        return ans

    return clean_math_string(ans)


def re_extract_and_save_json():
    print(f"Loading raw results from {INPUT_FILE}...\n")

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    if MAX_ID != -1:
        data = [item for item in data if item.get("id", 0) <= MAX_ID]

    total = len(data)
    if total == 0:
        print("No samples found.")
        return

    new_correct = 0
    failed_extractions = 0
    output_data = []

    for item in data:
        q_id = item.get("id", "Unknown")
        raw_output = item.get("raw_output", "")
        raw_gt = item.get("ground_truth", "")

        # Process the strings
        clean_gt = clean_math_string(raw_gt)
        new_extracted = extract_model_answer(raw_output)

        is_correct = new_extracted == clean_gt

        if is_correct:
            new_correct += 1
        elif new_extracted == "Extraction Failed":
            failed_extractions += 1

        # Only save the essential fields to the new JSON
        output_data.append(
            {
                "id": q_id,
                "ground_truth": clean_gt,
                "new_extracted": new_extracted,
                "is_correct": is_correct,
            }
        )

    # Calculate Metrics
    accuracy = (new_correct / total) * 100
    old_accuracy = (sum(1 for d in data if d.get("is_correct", False)) / total) * 100

    print("=" * 50)
    print("FINAL EXTRACTION BENCHMARK")
    print("=" * 50)
    print(f"Total Samples:      {total}")
    print(f"Old Accuracy:       {old_accuracy:.2f}%")
    print(f"New Accuracy:       {accuracy:.2f}%")
    print(f"Failed Extractions: {failed_extractions}")
    print("=" * 50)

    # Save to new JSON
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4)

    print(f"\n✅ Cleaned extraction data saved successfully to:\n{OUTPUT_FILE}")


## Execution

In [ ]:
if __name__ == "__main__":
    re_extract_and_save_json()
